# ⚡ Uplift Modeling — Incrementalidad de Campaña
> **Caso de negocio:** Banco / Financiera · Campaña de retención  
> **Framework:** CRISP-DM · Digital Marketing Analytics — UPC  
> **Autor:** Miguel Salazar · Attach Group

---

## ¿Qué es Uplift Modeling?

El uplift (o *incrementalidad*) mide el **efecto causal de una intervención** — no solo quién convierte, sino quién convierte **porque** recibió la campaña.

### Los 4 tipos de clientes:

| Tipo | Descripción | Acción |
|------|-------------|--------|
| **Persuadible** | Solo convierte si recibe campaña | ✅ TARGET principal |
| **Always buy** | Convierte igual, con o sin campaña | ⚠️ Gastar en ellos es malgastar |
| **Never buy** | No convierte en ningún caso | ⚠️ No vale la pena invertir |
| **Sleeping dog** | La campaña tiene efecto negativo | ❌ Evitar activamente |

---

## Contexto del problema

Una campaña de retención bancaria cuesta S/.180K/mes. El análisis revela que el **40% del presupuesto** se gasta en clientes que habrían convertido igual (*always buy*), sin ningún impacto incremental.

**Objetivo:** Identificar los clientes *persuadibles* para concentrar el presupuesto donde genera impacto real.

**Algoritmo:** Two-model approach — `uplift = P(conv|tratado) − P(conv|control)`

## 📦 Fase 0 — Setup

In [ ]:
!pip install -q plotly scikit-uplift

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
print('Setup completo ✓')

## 1️⃣ Fase 1 — Business Understanding

In [ ]:
# Comparación: modelo de propensión tradicional vs uplift
print('=== PROPENSIÓN TRADICIONAL vs UPLIFT MODELING ===')
print()
print('Propensión tradicional:')
print('  Pregunta: ¿Quién tiene más probabilidad de convertir?')
print('  Problema: Incluye "always buyers" — gastan sin impacto incremental')
print('  Resultado: Activas a clientes que ya iban a comprar solos')
print()
print('Uplift Modeling:')
print('  Pregunta: ¿Quién convierte PORQUE recibió la campaña?')
print('  Ventaja:  Filtra "always buyers" y "sleeping dogs"')
print('  Resultado: Presupuesto solo en clientes persuadibles')
print()
print('KPIs de éxito:')
print('  ATE (Average Treatment Effect) >= 0.05')
print('  Uplift top 10% >= 0.15')
print('  Reducción de costo de campaña: -40% con mismo revenue')

## 2️⃣ Fase 2 — Data Understanding

In [ ]:
# Dataset con estructura de experimento controlado (A/B test)
# Requisito fundamental: asignación ALEATORIA al grupo tratado/control
N = 2000

edad          = np.random.randint(20, 65, N)
recencia      = np.random.randint(1, 91, N)
n_compras     = np.random.poisson(3, N).clip(0, 12)
ticket        = np.round(np.random.lognormal(1.5, 0.6, N), 1).clip(0.5, 20)
canal_digital = (np.random.random(N) > 0.4).astype(int)

# Asignación aleatoria: 50% tratado, 50% control
tratamiento   = (np.random.random(N) > 0.5).astype(int)

# Probabilidad base de conversión (sin campaña)
base_z    = -2.5 + 0.12*n_compras + 0.04*ticket - 0.01*recencia
base_prob = 1 / (1 + np.exp(-base_z))

# Efecto del tratamiento: heterogéneo — no todos responden igual
# Los más activos y digitales responden más
treatment_effect = (
    0.20 * (n_compras / 12)
    + 0.10 * canal_digital
    + np.random.normal(0, 0.05, N)
).clip(0, 0.5)

conv_prob = np.where(
    tratamiento == 1,
    np.minimum(base_prob + treatment_effect, 1.0),
    base_prob
)
convirtio = (np.random.random(N) < conv_prob).astype(int)

df = pd.DataFrame({
    'cliente_id':    range(1, N+1),
    'edad':          edad,
    'recencia_dias': recencia,
    'n_compras_prev': n_compras,
    'ticket_prom_k': ticket,
    'canal_digital': canal_digital,
    'tratamiento':   tratamiento,
    'convirtio':     convirtio
})

print(f'Dataset: {df.shape}')
print(f'Grupo tratado:  {tratamiento.sum()} clientes ({tratamiento.mean():.0%})')
print(f'Grupo control:  {(1-tratamiento).sum()} clientes ({(1-tratamiento).mean():.0%})')
df.head()

In [ ]:
# Análisis de la conversión por grupo — verificación del experimento
summary = df.groupby('tratamiento').agg(
    n=('convirtio', 'count'),
    conversiones=('convirtio', 'sum'),
    tasa_conversion=('convirtio', 'mean')
).round(4)
summary.index = ['Control (sin campaña)', 'Tratado (con campaña)']
print('=== RESULTADO GLOBAL DEL EXPERIMENTO ===')
display(summary)

ate_simple = (summary.loc['Tratado (con campaña)', 'tasa_conversion']
              - summary.loc['Control (sin campaña)', 'tasa_conversion'])
print(f'\nATE simple (diferencia de medias): {ate_simple:+.4f} ({ate_simple:+.1%})')
print('→ La campaña genera este incremento promedio de conversión')

In [ ]:
# Heterogeneidad del efecto: el tratamiento no afecta igual a todos
fig = make_subplots(rows=1, cols=2,
                    subplot_titles=['Conv rate por n_compras_prev y tratamiento',
                                    'Conv rate por canal_digital y tratamiento'])

for feat, row, col in [('n_compras_prev', 1, 1), ('canal_digital', 1, 2)]:
    for trat_val, label, color in [(1,'Tratado','#0d9488'), (0,'Control','#c0392b')]:
        grp = df[df['tratamiento']==trat_val].groupby(feat)['convirtio'].mean().reset_index()
        fig.add_trace(
            go.Scatter(x=grp[feat], y=grp['convirtio'],
                       mode='lines+markers', name=label,
                       line=dict(color=color),
                       showlegend=(feat == 'n_compras_prev')),
            row=row, col=col
        )

fig.update_yaxes(tickformat='.0%')
fig.update_layout(height=380, title='Heterogeneidad del efecto de la campaña',
                  plot_bgcolor='white', paper_bgcolor='white')
fig.show()
print('Observación: el efecto de la campaña es mayor en clientes más activos (n_compras alto)')

## 3️⃣ Fase 3 — Data Preparation

In [ ]:
FEATURES = ['edad', 'recencia_dias', 'n_compras_prev', 'ticket_prom_k', 'canal_digital']
TREATMENT = 'tratamiento'
TARGET    = 'convirtio'

# Separar grupos para el two-model approach
treated = df[df[TREATMENT] == 1].copy()
control = df[df[TREATMENT] == 0].copy()

print(f'Grupo tratado:  {len(treated)} registros | Conv rate: {treated[TARGET].mean():.1%}')
print(f'Grupo control:  {len(control)} registros | Conv rate: {control[TARGET].mean():.1%}')

# Scaler ajustado en todos los datos (para consistencia en producción)
scaler = StandardScaler()
X_all = scaler.fit_transform(df[FEATURES].fillna(0))
X_t   = scaler.transform(treated[FEATURES].fillna(0))
X_c   = scaler.transform(control[FEATURES].fillna(0))
y_t   = treated[TARGET].astype(int)
y_c   = control[TARGET].astype(int)

print('\nPreparación completada ✓')

## 4️⃣ Fase 4 — Modeling

### Two-model approach

```
Modelo T → entrenado SOLO en grupo tratado → P(conv | campaña)
Modelo C → entrenado SOLO en grupo control → P(conv | sin campaña)

Uplift score = P(conv | campaña) − P(conv | sin campaña)
```

**Ventaja:** Simple e interpretable.  
**Limitación:** Los errores de ambos modelos se acumulan en la diferencia.

In [ ]:
# Modelo T: aprende el comportamiento del grupo que recibió campaña
model_t = LogisticRegression(C=1.0, max_iter=300, random_state=42)
model_t.fit(X_t, y_t)
auc_t = roc_auc_score(y_t, model_t.predict_proba(X_t)[:, 1])
print(f'Modelo T (tratados) — AUC (train): {auc_t:.3f}')

# Modelo C: aprende el comportamiento del grupo que NO recibió campaña
model_c = LogisticRegression(C=1.0, max_iter=300, random_state=42)
model_c.fit(X_c, y_c)
auc_c = roc_auc_score(y_c, model_c.predict_proba(X_c)[:, 1])
print(f'Modelo C (control)  — AUC (train): {auc_c:.3f}')

# Calcular uplift score para todos los clientes
p_treated = model_t.predict_proba(X_all)[:, 1]
p_control = model_c.predict_proba(X_all)[:, 1]
uplift_scores = p_treated - p_control

df['p_con_campaña']    = p_treated
df['p_sin_campaña']    = p_control
df['uplift_score']     = uplift_scores

print(f'\nUplift score — estadísticas:')
print(df['uplift_score'].describe().round(4))

## 5️⃣ Fase 5 — Evaluation

In [ ]:
# Métricas de incrementalidad
ate         = uplift_scores.mean()
top10_n     = max(1, len(uplift_scores) // 10)
top10_idx   = np.argsort(uplift_scores)[::-1][:top10_n]
top10_lift  = uplift_scores[top10_idx].mean()

# Qini coefficient: cuánto mejor que random es el ranking del modelo
rank       = np.argsort(uplift_scores)[::-1]
cum_conv   = np.cumsum(df[TARGET].values[rank])
rand_line  = np.linspace(0, cum_conv[-1], len(cum_conv))
qini       = float((cum_conv - rand_line).sum() / len(cum_conv))

criterios = {'ATE': 0.05, 'Top 10% lift': 0.15}
resultados = {'ATE': ate, 'Top 10% lift': top10_lift, 'Qini': qini}

print('=== MÉTRICAS DE UPLIFT ===')
for k, v in resultados.items():
    umbral = criterios.get(k)
    estado = ''
    if umbral:
        estado = '✅ APROBADO' if v >= umbral else f'❌ (umbral {umbral:.2f})'
    print(f'  {k:18s}: {v:+.4f}  {estado}')

In [ ]:
# Uplift por decil: el modelo debe rankear bien (deciles altos = mayor uplift)
df['decil'] = pd.qcut(df['uplift_score'], 10,
                       labels=[f'D{i}' for i in range(1, 11)],
                       duplicates='drop')

decile_stats = df.groupby('decil', observed=True).agg(
    n=('uplift_score', 'count'),
    uplift_medio=('uplift_score', 'mean'),
    conv_tratados=('convirtio', lambda x: x[df.loc[x.index,'tratamiento']==1].mean() if (df.loc[x.index,'tratamiento']==1).sum() > 0 else 0),
    conv_control=('convirtio', lambda x: x[df.loc[x.index,'tratamiento']==0].mean() if (df.loc[x.index,'tratamiento']==0).sum() > 0 else 0)
).reset_index()

colors = ['#0d9488' if u > 0 else '#c0392b' for u in decile_stats['uplift_medio']]
fig = go.Figure(go.Bar(
    x=decile_stats['decil'].astype(str),
    y=decile_stats['uplift_medio'],
    marker_color=colors,
    text=[f'{u:+.3f}' for u in decile_stats['uplift_medio']],
    textposition='outside'
))
fig.add_hline(y=ate, line_dash='dash', line_color='orange',
              annotation_text=f'ATE = {ate:+.3f}')
fig.update_layout(
    title='Uplift medio por decil (D10 = mayor incrementalidad)',
    xaxis_title='Decil', yaxis_title='Uplift score medio',
    height=400, plot_bgcolor='white', paper_bgcolor='white',
    yaxis=dict(zeroline=True, zerolinecolor='#888', gridcolor='#f0f0f0')
)
fig.show()
print('Interpretación: D10 son los más persuadibles — enfocar campaña aquí')

In [ ]:
# Distribución de uplift scores por grupo
fig = go.Figure()
for trat_val, label, color in [(1,'Grupo tratado','#1a4c8c'), (0,'Grupo control','#6b7280')]:
    sub = df[df['tratamiento'] == trat_val]['uplift_score']
    fig.add_trace(go.Histogram(x=sub, name=label, opacity=0.7,
                               marker_color=color, nbinsx=30))
fig.add_vline(x=0, line_color='red', line_dash='dash', annotation_text='uplift=0')
fig.update_layout(
    barmode='overlay',
    title='Distribución de uplift scores (positivo = persuadible)',
    xaxis_title='Uplift score', yaxis_title='Frecuencia',
    height=380, plot_bgcolor='white', paper_bgcolor='white'
)
fig.show()

In [ ]:
# Curva de Qini: benchmark del modelo contra selección aleatoria
# Eje X: fracción de la población contactada (ordenada por uplift desc)
# Eje Y: conversiones incrementales acumuladas

rank = np.argsort(uplift_scores)[::-1]
fraccion = np.linspace(0, 1, len(rank))
cum_conv_model = np.cumsum(df[TARGET].values[rank]) / df[TARGET].sum()
cum_conv_random = np.linspace(0, 1, len(rank))

fig = go.Figure()
fig.add_trace(go.Scatter(x=fraccion, y=cum_conv_model, mode='lines',
                          name=f'Modelo (Qini={qini:.2f})',
                          line=dict(color='#1a4c8c', width=2.5)))
fig.add_trace(go.Scatter(x=fraccion, y=cum_conv_random, mode='lines',
                          name='Aleatorio', line=dict(color='#ccc', dash='dash')))
fig.add_vline(x=0.2, line_dash='dot', line_color='#0d9488',
              annotation_text='Top 20% de clientes')
fig.update_layout(
    title='Curva Qini — Conversiones incrementales acumuladas',
    xaxis_title='Fracción de clientes contactados',
    yaxis_title='Fracción de conversiones capturadas',
    height=420, plot_bgcolor='white', paper_bgcolor='white',
    xaxis=dict(tickformat='.0%', gridcolor='#f0f0f0'),
    yaxis=dict(tickformat='.0%', gridcolor='#f0f0f0')
)
fig.show()

# Cuántas conversiones captura el top 20%
idx_20 = int(len(cum_conv_model) * 0.2)
print(f'Con el top 20% (modelo): captura {cum_conv_model[idx_20]:.0%} de las conversiones')
print(f'Con el top 20% (random): captura {cum_conv_random[idx_20]:.0%} de las conversiones')
print(f'Lift del modelo en top 20%: {cum_conv_model[idx_20]/cum_conv_random[idx_20]:.1f}x')

In [ ]:
# Scatter: segmentación de clientes por tipo
df['tipo_cliente'] = 'Never buy'
df.loc[(df['p_sin_campaña'] >= 0.5) & (df['uplift_score'] < 0.05), 'tipo_cliente'] = 'Always buy'
df.loc[df['uplift_score'] >= 0.15, 'tipo_cliente'] = 'Persuadible'
df.loc[df['uplift_score'] < -0.02, 'tipo_cliente'] = 'Sleeping dog'

print('Distribución de tipos de cliente:')
print(df['tipo_cliente'].value_counts())
print(f'\nPresupuesto recomendado: enfocar en {(df["tipo_cliente"]=="Persuadible").sum()} persuadibles ({(df["tipo_cliente"]=="Persuadible").mean():.0%})')

## 6️⃣ Fase 6 — Deployment

In [ ]:
# Segmentación final para targeting
df['segmento_uplift'] = pd.cut(
    df['uplift_score'],
    bins=[-1, 0.02, 0.10, 1],
    labels=['No persuadible', 'Responde algo', 'Persuadible']
)

print('=== AUDIENCIA SEGMENTADA PARA CAMPAÑA ===')
seg_stats = df.groupby('segmento_uplift', observed=True).agg(
    n_clientes=('cliente_id', 'count'),
    uplift_medio=('uplift_score', 'mean'),
    conv_rate=('convirtio', 'mean')
).round(4)
display(seg_stats)

n_persuadibles = (df['segmento_uplift'] == 'Persuadible').sum()
n_total = len(df)
print(f'\nContactar solo persuadibles: {n_persuadibles} de {n_total} ({n_persuadibles/n_total:.0%})')
print(f'Ahorro en campaña: {1 - n_persuadibles/n_total:.0%} menos contactos')

In [ ]:
# Guardar outputs
import joblib

output_cols = ['cliente_id','p_con_campaña','p_sin_campaña','uplift_score','segmento_uplift']
df[output_cols].sort_values('uplift_score', ascending=False).to_csv(
    'audiencia_uplift_scored.csv', index=False
)

joblib.dump({'model_t': model_t, 'model_c': model_c, 'scaler': scaler},
            'modelos_uplift.pkl')
print('Outputs guardados ✓')

print('\n=== RESUMEN EJECUTIVO ===')
costo_por_contacto = 12  # S/.
costo_total_masivo   = n_total * costo_por_contacto
costo_uplift         = n_persuadibles * costo_por_contacto
ahorro               = costo_total_masivo - costo_uplift
print(f'Costo campaña masiva:   S/.{costo_total_masivo:,.0f}')
print(f'Costo campaña uplift:   S/.{costo_uplift:,.0f}')
print(f'Ahorro:                 S/.{ahorro:,.0f} ({ahorro/costo_total_masivo:.0%})')
print(f'ATE del modelo:         {ate:+.4f} ({ate:+.1%} de incremento en conversión)')
print(f'\nArquitectura de producción:')
print('  Escora mensual en BigQuery → filtra persuadibles → CDP → campaña')

In [ ]:
# Bonus: función de predicción para producción
def predict_uplift(cliente_features: dict,
                   model_t, model_c, scaler,
                   feature_names: list) -> dict:
    """Calcula el uplift score de un cliente individual.

    Args:
        cliente_features: dict con los valores de cada feature
        model_t:          modelo entrenado en grupo tratado
        model_c:          modelo entrenado en grupo control
        scaler:           StandardScaler ajustado en entrenamiento
        feature_names:    lista de features en el orden correcto

    Returns:
        dict con p_con_campana, p_sin_campana, uplift, segmento
    """
    X = np.array([[cliente_features[f] for f in feature_names]])
    X_s = scaler.transform(X)

    p_t = model_t.predict_proba(X_s)[0][1]
    p_c = model_c.predict_proba(X_s)[0][1]
    uplift = p_t - p_c

    if uplift >= 0.10:
        segmento = 'Persuadible'
    elif uplift >= 0.02:
        segmento = 'Responde algo'
    else:
        segmento = 'No persuadible'

    return {
        'p_con_campaña':  round(p_t, 4),
        'p_sin_campaña':  round(p_c, 4),
        'uplift_score':   round(uplift, 4),
        'segmento':       segmento
    }

# Ejemplo de uso
cliente_ejemplo = {
    'edad': 35,
    'recencia_dias': 10,
    'n_compras_prev': 6,
    'ticket_prom_k': 8.5,
    'canal_digital': 1
}

resultado = predict_uplift(cliente_ejemplo, model_t, model_c, scaler, FEATURES)
print('=== PREDICCIÓN INDIVIDUAL ===')
for k, v in resultado.items():
    print(f'  {k}: {v}')